# AeroScientia — Level 1: NLP Corpus Exploration

**Objetivo:** Explorar el corpus técnico aeroespacial, visualizar la distribución de entidades,
y validar el pipeline de extracción antes del fine-tuning.

**Pipeline:**
1. Load corpus
2. Analyze entity distribution
3. Run rule-based extraction demo
4. Visualize bilingual coverage
5. Prepare data for NER training


In [ ]:
# Setup
import sys
sys.path.insert(0, '..')

import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import Counter
from pathlib import Path

# AeroScientia modules
from corpus.aerospace_corpus import CORPUS_TEXTS, CORPUS_ES
from bilingual_glossary import GLOSSARY, ENTITY_TYPES, DOMAINS, search_glossary
from extractor import AeroExtractor, DEMO_TEXT

# Style
plt.style.use('dark_background')
AMBER = '#F5A623'
CYAN  = '#00D4FF'
MARS  = '#FF6B35'

print('✓ Imports OK')
print(f'  Corpus EN: {len(CORPUS_TEXTS)} documents')
print(f'  Corpus ES: {len(CORPUS_ES)} documents')
print(f'  Glossary:  {len(GLOSSARY)} terms')


## 1. Corpus Overview — ¿Qué tenemos?

In [ ]:
# Build a dataframe of all annotated entities
rows = []
for doc in CORPUS_TEXTS + CORPUS_ES:
    for entity_text, entity_type in doc.get('entities', []):
        rows.append({
            'doc_id':      doc['id'],
            'domain':      doc['domain'],
            'source_lang': doc['lang'],
            'entity_text': entity_text,
            'entity_type': entity_type,
            'text_length': len(entity_text),
        })

df = pd.DataFrame(rows)
print(f'Total annotated entities: {len(df)}')
print(f'Unique entity texts:      {df.entity_text.nunique()}')
print(f'Entity types:             {df.entity_type.nunique()}')
print(f'Domains:                  {df.domain.nunique()}')
df.head(10)


In [ ]:
# Entity type distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6), facecolor='#0a0a0c')

# --- Entity types ---
type_counts = df.entity_type.value_counts()
colors = [AMBER if i < 3 else CYAN for i in range(len(type_counts))]

axes[0].barh(type_counts.index, type_counts.values, color=colors, edgecolor='none')
axes[0].set_facecolor('#111118')
axes[0].set_title('Entity Type Distribution', color=AMBER, fontsize=13, pad=15)
axes[0].tick_params(colors='#aaa', labelsize=9)
axes[0].set_xlabel('Count', color='#888')
axes[0].invert_yaxis()

# --- Domain distribution ---
domain_counts = df.domain.value_counts()
domain_colors = [AMBER, CYAN, MARS, '#8B5CF6', '#34D399']

wedges, texts, autotexts = axes[1].pie(
    domain_counts.values,
    labels=domain_counts.index,
    colors=domain_colors[:len(domain_counts)],
    autopct='%1.1f%%',
    pctdistance=0.8,
    startangle=90,
    wedgeprops={'linewidth': 2, 'edgecolor': '#0a0a0c'}
)
for text in texts + autotexts:
    text.set_color('white')
    text.set_fontsize(9)
axes[1].set_facecolor('#111118')
axes[1].set_title('Domain Distribution', color=AMBER, fontsize=13, pad=15)

fig.patch.set_facecolor('#0a0a0c')
plt.tight_layout()
plt.savefig('../output/corpus_distribution.png', dpi=150, bbox_inches='tight',
            facecolor='#0a0a0c')
plt.show()
print('✓ Saved: output/corpus_distribution.png')


## 2. Bilingual Glossary Coverage

In [ ]:
# Build glossary dataframe
g_df = pd.DataFrame(GLOSSARY)
print(f'Glossary size: {len(g_df)} terms')
print(f'Domains covered:')
print(g_df.domain.value_counts())
print(f'\nEntity types covered:')
print(g_df.entity_type.value_counts())


In [ ]:
# Show sample glossary entries per domain
for domain in g_df.domain.unique():
    print(f'\n── {domain.upper()} ──────────────────────────────────────')
    subset = g_df[g_df.domain == domain][['en', 'es', 'entity_type', 'symbol']].head(5)
    print(subset.to_string(index=False))


## 3. Rule-Based Extractor Demo

In [ ]:
# Run extractor on demo text
extractor = AeroExtractor(force_rules=True)
result = extractor.extract_text(DEMO_TEXT, source='demo_notebook')

extractor.print_report(result)

# Save outputs
Path('../output').mkdir(exist_ok=True)
extractor.export_json(result, '../output/notebook_extraction.json')
glossary = extractor.export_glossary_json(result, '../output/notebook_glossary.json')
df_result = extractor.export_csv(result, '../output/notebook_extraction.csv')


In [ ]:
# Visualize extracted entities
if df_result is not None:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5), facecolor='#0a0a0c')

    # Entity types
    type_dist = df_result.entity_type.value_counts()
    axes[0].barh(type_dist.index, type_dist.values, color=CYAN, alpha=0.8)
    axes[0].set_facecolor('#111118')
    axes[0].set_title('Extracted: Entity Types', color=CYAN, fontsize=12)
    axes[0].tick_params(colors='#aaa', labelsize=9)
    axes[0].invert_yaxis()

    # Domain distribution
    domain_dist = df_result.domain.value_counts()
    axes[1].bar(domain_dist.index, domain_dist.values,
                color=[AMBER, CYAN, MARS, '#8B5CF6', '#34D399'][:len(domain_dist)])
    axes[1].set_facecolor('#111118')
    axes[1].set_title('Extracted: Domain Distribution', color=AMBER, fontsize=12)
    axes[1].tick_params(colors='#aaa', labelsize=9)
    axes[1].tick_params(axis='x', rotation=30)

    fig.patch.set_facecolor('#0a0a0c')
    plt.tight_layout()
    plt.savefig('../output/extraction_results.png', dpi=150, bbox_inches='tight',
                facecolor='#0a0a0c')
    plt.show()


## 4. Bilingual Term Alignment — Sample Output

In [ ]:
# Show bilingual glossary output
with open('../output/notebook_glossary.json') as f:
    extracted_glossary = json.load(f)

print(f'Bilingual terms extracted: {len(extracted_glossary)}\n')

g_out = pd.DataFrame(extracted_glossary)
display_cols = ['en', 'es', 'entity_type', 'domain', 'symbol']
display_cols = [c for c in display_cols if c in g_out.columns]

# Group by domain
for domain in g_out.domain.unique():
    print(f'\n{'─'*70}')
    print(f'  {domain.upper()}')
    print(f'{'─'*70}')
    subset = g_out[g_out.domain == domain][display_cols]
    for _, row in subset.iterrows():
        print(f'  EN: {row["en"]}')
        print(f'  ES: {row["es"]}')
        if row.get('symbol'):
            print(f'  ∑:  {row["symbol"]}')
        print()


## 5. Test Custom Text

In [ ]:
# Test the extractor on your own text
MY_TEXT = """
Paste any aerospace technical text here and the extractor will identify
equations, laws, parameters, and terminology — and align them with
the bilingual EN/ES glossary automatically.

For example: The drag polar CD = CD0 + CL^2/(pi*e*AR) shows how induced drag 
increases with lift coefficient. At hypersonic speeds (Mach > 5), aerodynamic 
heating Q = 1/2 * rho * v^3 becomes the primary design constraint for the 
thermal protection system (TPS). The Tsiolkovsky rocket equation Delta-v = ve * ln(m0/mf) 
and Kepler's laws together define the complete mission design space.
"""

result = extractor.extract_text(MY_TEXT, source='custom_test')
extractor.print_report(result)

# Show as dataframe
from dataclasses import asdict
df_custom = pd.DataFrame([asdict(e) for e in result.entities])
if not df_custom.empty:
    cols = ['text', 'entity_type', 'domain', 'confidence', 'translation_es']
    print(df_custom[[c for c in cols if c in df_custom.columns]].to_string())


## 6. Next Steps — Preparar Training Data para NER

In [ ]:
# Preview training data for NER fine-tuning
import sys
sys.path.insert(0, '..')
from ner_trainer import TrainingDataBuilder, TrainingConfig

config = TrainingConfig()
builder = TrainingDataBuilder(config)
training_data = builder.build_training_examples()
train, eval_d = builder.split_train_eval(training_data)

print(f'\n✓ Training data ready:')
print(f'  Train examples: {len(train)}')
print(f'  Eval examples:  {len(eval_d)}')
print(f'\nTo train the model, run in terminal:')
print(f'  python ner_trainer.py --train')
print(f'  python ner_trainer.py --eval')
